<a href="https://colab.research.google.com/github/williamng252/BT_tri_tue_nhan_tao/blob/main/8puzzle_with_model_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BÀI TẬP: GIẢI QUYẾT 8-PUZZLE BẰNG MODEL-BASED REFLEX AGENT

## 1. Phân tích các thành phần của Agent
Trong bài toán 8-Puzzle, Tác tử phản xạ dựa trên mô hình (Model-Based Reflex Agent) được định nghĩa với các thành phần sau:
* **`state` (Trạng thái hiện tại):** Là vị trí của các con số trên ma trận 3x3 hiện tại.
* **`model` (Mô hình trí nhớ):** Là một tập hợp (`set`) lưu trữ tất cả các trạng thái ma trận mà Agent ĐÃ ĐI QUA. Điều này giúp Agent không bị đi lùi hoặc kẹt vào vòng lặp vô hạn.
* **`rules` (Tập luật & Hàm Heuristic):** - *Luật sinh thái:* Ô trống (0) chỉ được di chuyển Lên, Xuống, Trái, Phải nếu không đụng tường.
    - *Luật đánh giá (Heuristic):* Trong các nước đi hợp lệ, ưu tiên chọn nước đi tạo ra trạng thái có **số lượng ô sai vị trí so với đích là ít nhất**.

## 2. Ánh xạ vào mã giả (Pseudocode)
1. `state <- UPDATE-STATE(...)`: Agent nhận diện ma trận hiện tại và thêm nó vào `model` (bộ nhớ).
2. `rule <- RULE-MATCH(state, rules)`: Agent tính toán các nước đi có thể, loại bỏ các nước đi đã nằm trong `model`, và chọn ra nước đi có điểm Heuristic tốt nhất.
3. `action <- rule.ACTION`: Thực hiện hoán đổi ô trống theo hướng đã chọn.

In [2]:
import time
from typing import List, Optional

START = [1, 2, 3, 4, 0, 6, 7, 5, 8]
GOAL  = [1, 2, 3, 4, 5, 6, 7, 8, 0]

class ModelBased8Puzzle:
    def __init__(self, goal: List[int]):
        self.goal = goal
        self.model = set()  # BỘ NHỚ: Lưu các trạng thái đã qua (persistent model)

    def print_board(self, state: List[int], step: int, action: str):
        print(f"--- BƯỚC {step} {action} ---")
        for i in range(0, 9, 3):
            print(f"| {state[i]} {state[i+1]} {state[i+2]} |")
        print("-" * 13)

    def heuristic_rule(self, state: List[int]) -> int:
        """TẬP LUẬT (RULES): Đếm số lượng ô đang sai vị trí so với Đích"""
        return sum(1 for i in range(9) if state[i] != self.goal[i] and state[i] != 0)

    def RULE_MATCH(self, current_state: List[int]) -> Optional[dict]:
        """
        Tương đương: rule <- RULE-MATCH(state, rules)
        Nhiệm vụ: Sinh các bước đi, đối chiếu luật (Heuristic) và chọn bước tốt nhất.
        """
        blank_idx = current_state.index(0)
        row, col = divmod(blank_idx, 3)
        directions = {"Lên": (-1, 0), "Xuống": (1, 0), "Trái": (0, -1), "Phải": (0, 1)}
        choices = []

        for move_name, (dr, dc) in directions.items():
            nr, nc = row + dr, col + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                target_idx = nr * 3 + nc
                new_state = list(current_state)
                # Thử thực hiện hành động
                new_state[blank_idx], new_state[target_idx] = new_state[target_idx], new_state[blank_idx]

                state_tuple = tuple(new_state)

                # ÁP DỤNG MODEL: Chỉ lấy luật nếu trạng thái này chưa từng đi qua
                if state_tuple not in self.model:
                    score = self.heuristic_rule(new_state)
                    choices.append({
                        "action": move_name,
                        "target_idx": target_idx,
                        "score": score,
                        "new_state": state_tuple
                    })

        # Chọn ra luật có hành động tốt nhất (điểm sai lệch nhỏ nhất)
        choices.sort(key=lambda x: x["score"])
        return choices[0] if choices else None

    def execute(self, percept_state: List[int]):
        """
        Tương đương: function MODEL-BASED-REFLEX-AGENT(percept)
        """
        # 1. state <- UPDATE-STATE(state, action, percept, model)
        # Ghi nhớ trạng thái hiện tại vào bộ não (model)
        self.model.add(tuple(percept_state))

        # 2. rule <- RULE-MATCH(state, rules)
        best_rule = self.RULE_MATCH(percept_state)

        # 3. action <- rule.ACTION
        if best_rule:
            return best_rule # Trả về thông tin hành động để thực thi
        return None

if __name__ == "__main__":
    current_state = list(START)
    agent = ModelBased8Puzzle(GOAL)

    agent.print_board(current_state, 0, "[KHỞI ĐỘNG]")

    step = 0
    max_steps = 50

    while current_state != GOAL and step < max_steps:
        step += 1

        # Đưa trạng thái ma trận hiện tại (percept) cho Agent
        rule_action = agent.execute(current_state)

        if rule_action is None:
            print("THẤT BẠI: AI bị kẹt, không còn đường đi mới!")
            break

        # Áp dụng hành động ra môi trường thực
        move_idx = rule_action["target_idx"]
        action_name = rule_action["action"]
        z_idx = current_state.index(0)

        # Hoán đổi vị trí
        current_state[z_idx], current_state[move_idx] = current_state[move_idx], current_state[z_idx]

        agent.print_board(current_state, step, f"(Đi '{action_name}')")
        time.sleep(0.2) # Chạy chậm lại để xem trên Jupyter

    if current_state == GOAL:
        print("KẾT QUẢ: Thành công!")

--- BƯỚC 0 [KHỞI ĐỘNG] ---
| 1 2 3 |
| 4 0 6 |
| 7 5 8 |
-------------
--- BƯỚC 1 (Đi 'Xuống') ---
| 1 2 3 |
| 4 5 6 |
| 7 0 8 |
-------------
--- BƯỚC 2 (Đi 'Phải') ---
| 1 2 3 |
| 4 5 6 |
| 7 8 0 |
-------------
KẾT QUẢ: Thành công!


Link git: https://github.com/williamng252/BT_tri_tue_nhan_tao.git